# How to prompt: A prompting tutorial for modern Agentic AI

Welcome to Agentic AI. Traditional prompting relies on zero-shot or few-shot examples to get a direct answer. 

**Agentic Prompting** relies on a structured loop. We prompt the LLM to act as a state machine. It must read the user's input, formulate a **Thought**, select an **Action** (a tool), and wait for an **Observation** (the tool's output). It loops this process until it has enough information to formulate a **Final Answer**.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing LangChain and configuring the LLM.
2. **Tool Definition:** Giving the LLM Python functions it can actually execute.
3. **The ReAct Prompt:** Analytically breaking down the exact text structure that forces an LLM to become an Agent.
4. **Agent Execution:** Running the autonomous loop.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, run: !pip install -q langchain langchain-openai python-dotenv

import os
from langchain_openai import ChatOpenAI
from langchain.agents import tool, AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# For this tutorial, you would normally need an OpenAI API key.
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Initialize the reasoning engine (the LLM)
# We set temperature=0 because we want analytical, deterministic reasoning, not creative hallucination.
llm = ChatOpenAI(model="gpt-4", temperature=0)

print("Environment configured and LLM reasoning engine initialized.")

## Step 1: Defining Tools (The Agent's Hands)

An LLM is just a brain locked in a box. To make it "Agentic", we must give it tools. 

In LangChain, we use the `@tool` decorator. 
**Analytical Rule:** The docstring (the text inside the `""" """`) is the most important part of this code. The LLM does not read your Python logic; it reads the docstring. Your prompt inside the docstring must explicitly explain *when* and *how* to use the tool.

In [ ]:
# Cell 4: Defining the Tools

@tool
def calculate_string(expression: str) -> str:
    """
    Useful for when you need to answer questions about math.
    Input should be a mathematical expression that the Python `eval()` function can execute.
    Example: '12 * (3 + 4)'
    """
    try:
        # Warning: eval() is unsafe for production without strict sanitization.
        # We use it here purely for mathematical demonstration.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error calculating: {e}"

@tool
def get_word_length(word: str) -> int:
    """
    Useful for when you need to find the length of a specific word.
    Input should be a single string.
    """
    return len(word)

# We package our tools into a list so we can inject them into the prompt later
tools = [calculate_string, get_word_length]

print(f"Defined {len(tools)} tools for the Agent.")

## Step 2: The ReAct Prompt Template (The Agent's Brain)

This is the secret sauce. To make an LLM autonomous, we don't just ask it a question. We provide it with a strict syntactical framework. 

We explicitly tell the LLM that it has access to tools, and we mathematically force its output to match a specific regex pattern (`Thought:`, `Action:`, `Action Input:`). When the LLM outputs `Action:`, the LangChain wrapper pauses the LLM, runs the Python tool, injects the `Observation:`, and starts the LLM again.

In [ ]:
# Cell 6: The ReAct Prompt Structure

# This is the industry-standard ReAct prompt pattern.
# Notice how strictly it defines the output format.
react_prompt_template = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format strictly:

Question: the input question you must answer
Thought: you should always think about what to do next
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(react_prompt_template)

print("ReAct Prompt Template successfully constructed.")

## Step 3: Constructing and Executing the Agent

Now we bind the three components together:
1. The **LLM** (The logic engine).
2. The **Tools** (The actionable functions).
3. The **Prompt** (The behavioral instructions).

We wrap these in an `AgentExecutor`. The executor acts as the `while` loop, continuously parsing the LLM's text, calling the Python functions, and feeding the data back into the `agent_scratchpad` variable until the LLM writes "Final Answer".

In [ ]:
# Cell 8: Execution and Tracing

# Create the specific ReAct agent logic
agent = create_react_agent(llm, tools, prompt)

# The AgentExecutor acts as the runtime loop
# verbose=True allows us to see the LLM's internal "Thought" process printed to the console
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True # If the LLM hallucinates formatting, this forces it to try again
)

# A complex prompt that requires multi-step reasoning and multiple tools
user_query = "What is the length of the word 'Supercalifragilisticexpialidocious' multiplied by 3.14?"

print("--- Starting Autonomous Agent Loop ---")
# Execute the loop
# In a live environment, this will trigger the API and output the Thought/Action logs
# response = agent_executor.invoke({"input": user_query})

print(f"\nUser Query: {user_query}")

# --- Simulated Trace Output ---
print("> Entering new AgentExecutor chain...")
print("Thought: I need to find the length of the word 'Supercalifragilisticexpialidocious' first.")
print("Action: get_word_length")
print("Action Input: Supercalifragilisticexpialidocious")
print("Observation: 34")
print("Thought: Now I need to multiply 34 by 3.14.")
print("Action: calculate_string")
print("Action Input: 34 * 3.14")
print("Observation: 106.76")
print("Thought: I now know the final answer.")
print("Final Answer: 106.76")
print("> Finished chain.")

## Conclusion

You have successfully constructed an Agentic AI pipeline!

**Key Analytical Takeaways:**
1. **Prompting is Programming:** In Agentic AI, the prompt is essentially compiling a runtime environment. By strictly enforcing the `Thought / Action / Observation` format, we "hack" the autoregressive nature of the LLM into becoming a Turing-complete loop.
2. **The Scratchpad:** The `{agent_scratchpad}` variable is crucial. It acts as the agent's short-term memory. Without it, the agent would forget what tool it just used and get trapped in an infinite loop.
3. **Semantic Tool Descriptions:** The LLM only knows how to use your Python functions based on your docstrings. If your agent is failing, the bug is rarely in your Python code; it is almost always an ambiguous description in your `@tool` prompt!